# DistilBERT Fine-tuning for Sentiment Classification (Colab GPU)

This notebook fine-tunes `distilbert-base-uncased` on the IMDb 50K dataset.
Run it on Google Colab with a GPU runtime for ~15 min training time.

**Setup:** Runtime → Change runtime type → GPU (T4)

In [ ]:
# Verify GPU is available
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU detected. Training will be very slow.')

In [ ]:
!pip install -q transformers kaggle beautifulsoup4

## 1. Download Dataset

Install kaggle and download the dataset directly.

In [ ]:
import os
from pathlib import Path
from getpass import getpass

os.environ['KAGGLE_USERNAME'] = input('Kaggle username: ')
os.environ['KAGGLE_KEY'] = getpass('Kaggle API key: ')

!pip install -q kaggle
!kaggle datasets download -d lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
!unzip -qo imdb-dataset-of-50k-movie-reviews.zip

print(f'Dataset downloaded: {Path("IMDB Dataset.csv").stat().st_size / 1e6:.1f} MB')

## 2. Preprocessing

In [ ]:
import re
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

RANDOM_SEED = 42

def clean_review(text):
    """Clean a single review: remove HTML, lowercase, strip noise."""
    text = BeautifulSoup(text, 'html.parser').get_text()
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print('Loading dataset...')
df = pd.read_csv('IMDB Dataset.csv')
print(f'Shape: {df.shape}')

print('Cleaning text...')
df['clean_review'] = [clean_review(r) for r in tqdm(df['review'])]
df['label'] = (df['sentiment'] == 'positive').astype(int)

# Same split as traditional models (80/10/10, seed=42)
X = df['clean_review']
y = df['label']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=RANDOM_SEED, stratify=y_temp
)

X_train = X_train.tolist()
X_val = X_val.tolist()
X_test = X_test.tolist()
y_train = y_train.tolist()
y_val = y_val.tolist()
y_test = y_test.tolist()

print(f'Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}')

## 3. Dataset & DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizer

MAX_LEN = 256
BATCH_SIZE = 32  # larger batch on GPU

class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long),
        }

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

train_dataset = ReviewDataset(X_train, y_train, tokenizer, MAX_LEN)
val_dataset = ReviewDataset(X_val, y_val, tokenizer, MAX_LEN)
test_dataset = ReviewDataset(X_test, y_test, tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, num_workers=2)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}')

## 4. Model Setup

In [ ]:
from transformers import DistilBertForSequenceClassification, get_linear_schedule_with_warmup

EPOCHS = 3
LEARNING_RATE = 2e-5
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=2
).to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps
)

print(f'Device: {DEVICE}')
print(f'Total training steps: {total_steps:,}')
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

## 5. Training Loop

In [ ]:
import time
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

def train_epoch(model, dataloader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch in tqdm(dataloader, desc='Training'):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = outputs.logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += len(labels)

    return total_loss / len(dataloader), correct / total


def evaluate(model, dataloader, device):
    model.eval()
    all_preds, all_probs, all_labels = [], [], []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Evaluating'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label']

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.softmax(outputs.logits, dim=1)
            preds = probs.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())
            all_labels.extend(labels.numpy())

    y_true = np.array(all_labels)
    y_pred = np.array(all_preds)
    y_prob = np.array(all_probs)

    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred),
        'recall': recall_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred),
        'roc_auc': roc_auc_score(y_true, y_prob),
        'predictions': y_pred,
        'probabilities': y_prob,
        'labels': y_true,
    }

In [ ]:
best_f1 = 0
history = []
start_time = time.time()

for epoch in range(EPOCHS):
    print(f'\n{"="*50}')
    print(f'Epoch {epoch+1}/{EPOCHS}')
    print(f'{"="*50}')

    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, DEVICE)
    print(f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}')

    val_metrics = evaluate(model, val_loader, DEVICE)
    print(f'Val Acc: {val_metrics["accuracy"]:.4f} | Val F1: {val_metrics["f1"]:.4f} | '
          f'Val ROC-AUC: {val_metrics["roc_auc"]:.4f}')

    history.append({
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'train_acc': train_acc,
        'val_accuracy': val_metrics['accuracy'],
        'val_f1': val_metrics['f1'],
        'val_roc_auc': val_metrics['roc_auc'],
    })

    if val_metrics['f1'] > best_f1:
        best_f1 = val_metrics['f1']
        # Save best model
        model.save_pretrained('distilbert_best')
        tokenizer.save_pretrained('distilbert_best')
        print(f'  >> New best model saved (F1: {best_f1:.4f})')

total_time = time.time() - start_time
print(f'\nTotal training time: {total_time:.0f}s ({total_time/60:.1f} min)')
print(f'Best validation F1: {best_f1:.4f}')

## 6. Test Set Evaluation

In [ ]:
# Load best model and evaluate on test set
best_model = DistilBertForSequenceClassification.from_pretrained('distilbert_best').to(DEVICE)

# Validation metrics (for comparison table -- matches traditional model output)
val_results = evaluate(best_model, val_loader, DEVICE)
print('Validation Set Results:')
print(f'  Accuracy:  {val_results["accuracy"]:.4f}')
print(f'  Precision: {val_results["precision"]:.4f}')
print(f'  Recall:    {val_results["recall"]:.4f}')
print(f'  F1:        {val_results["f1"]:.4f}')
print(f'  ROC-AUC:   {val_results["roc_auc"]:.4f}')

# Measure inference time on validation set (matches traditional model measurement)
inference_start = time.time()
_ = evaluate(best_model, val_loader, DEVICE)
inference_time_sec = round(time.time() - inference_start, 4)
print(f'\n  Inference time (val set, {len(X_val)} samples): {inference_time_sec:.4f}s')

# Model size on disk
model_size_bytes = sum(
    f.stat().st_size for f in Path('distilbert_best').rglob('*') if f.is_file()
)
model_size_mb = round(model_size_bytes / (1024 * 1024), 2)
print(f'  Model size: {model_size_mb:.2f} MB')

# Test metrics
test_results = evaluate(best_model, test_loader, DEVICE)
print(f'\nTest Set Results:')
print(f'  Accuracy:  {test_results["accuracy"]:.4f}')
print(f'  Precision: {test_results["precision"]:.4f}')
print(f'  Recall:    {test_results["recall"]:.4f}')
print(f'  F1:        {test_results["f1"]:.4f}')
print(f'  ROC-AUC:   {test_results["roc_auc"]:.4f}')

In [ ]:
# Save results as JSON for use in local project
# Fields match traditional model output exactly for side-by-side comparison
import json

results = {
    'model': 'distilbert',
    'accuracy': val_results['accuracy'],
    'precision': val_results['precision'],
    'recall': val_results['recall'],
    'f1': val_results['f1'],
    'roc_auc': val_results['roc_auc'],
    'train_time_sec': round(total_time, 2),
    'inference_time_sec': inference_time_sec,
    'model_size_mb': model_size_mb,
    'test_accuracy': test_results['accuracy'],
    'test_precision': test_results['precision'],
    'test_recall': test_results['recall'],
    'test_f1': test_results['f1'],
    'test_roc_auc': test_results['roc_auc'],
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'max_len': MAX_LEN,
    'history': history,
}

with open('transformer_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('Results saved to transformer_results.json')
print(f'\n--- Fields matching traditional model output ---')
print(f'model:              {results["model"]}')
print(f'accuracy:           {results["accuracy"]:.4f}')
print(f'precision:          {results["precision"]:.4f}')
print(f'recall:             {results["recall"]:.4f}')
print(f'f1:                 {results["f1"]:.4f}')
print(f'roc_auc:            {results["roc_auc"]:.4f}')
print(f'train_time_sec:     {results["train_time_sec"]}')
print(f'inference_time_sec: {results["inference_time_sec"]}')
print(f'model_size_mb:      {results["model_size_mb"]}')

## 7. Download Model

Zip the model and download it to your local machine. Then extract to `models/distilbert/` in the project.

In [ ]:
!zip -r distilbert_model.zip distilbert_best/ transformer_results.json

try:
    from google.colab import files
    files.download('distilbert_model.zip')
    print('Download started. Extract to models/distilbert/ in your local project.')
except ImportError:
    print('Not running in Colab. Manually download distilbert_model.zip.')

print(f'\nModel size:')
!du -sh distilbert_best/